In [1]:
from rag.models.google_genai_models import BaseGoogleModel

CHAT_PROMPT_TEMPLATE = f"""
Your are an helpful AI assistant. Your answer users messages in polite way.
Based on the chat history, provide your next message.

HISTORY: {{CONTEXT}}
LAST USER MESSAGE: {{QUERY}}
"""

class ChatModel(BaseGoogleModel):
    def __init__(self):
        super().__init__(prompt_template=CHAT_PROMPT_TEMPLATE)

chat_model = ChatModel()
res = chat_model.generate(query="Who are you?", context="`asda")
print(res)

/Users/wnowogorski/PycharmProjects/CHAT_AGH/rag/models/google_genai_models.py:12: LangChainPendingDeprecationWarning: langchain.indexes.prompts will be removed in the future.If you're relying on these prompts, please open an issue on GitHub to explain your use case.
  from rag.models.prompts import (


I am a helpful AI assistant. It's nice to meet you! How can I help you today?



In [2]:
from langchain_core.runnables import Runnable
from typing import Dict

class ChatRunnable(Runnable):
    def __init__(self, model: ChatModel):
        self.model = model

    def invoke(self, input: Dict[str, str]) -> str:
        query = input["query"]
        context = input.get("context", "")
        return self.model.generate(query=query, context=context)


In [3]:
from langchain.memory import ConversationBufferMemory
from langchain.storage import InMemoryStore  # or RedisStore, etc.

memory = ConversationBufferMemory(
    memory_key="context",
    input_key="query",
    return_messages=False,
)


/var/folders/hm/3hm9w85d5mz7h5ccmqhq1rzm0000gn/T/ipykernel_71540/2685812495.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [4]:
from langgraph.graph import StateGraph, END

from typing import TypedDict

class AgentState(TypedDict):
    query: str
    context: str
    response: str

def chat_node(state: AgentState) -> AgentState:
    input_query = state["query"]
    input_context = state.get("context", "")
    response = chat_model.generate(query=input_query, context=input_context)
    return {
        "query": input_query,
        "context": f'{input_context}\nUser: {input_query}\nAI: {response}',
        "response": response
    }

graph = StateGraph(AgentState)
graph.add_node("chat", chat_node)
graph.set_entry_point("chat")
graph.add_edge("chat", END)
agent_executor = graph.compile()


In [5]:
context = ""

def query(question, context):
    state = {"query": question, "context": context}
    result = agent_executor.invoke(state)
    context = result["context"]
    return result["response"], context


In [13]:
q = "Wymien wszystkie "

res, context = query(q, context)
print(context)


User: Who are you?
AI: As a large language model, I am developed in Google. I am trained on a massive amount of text data, which allows me to communicate and generate human-like text in response to a wide range of prompts and questions. How can I help you today?

User: What is the capital of Paris?
AI: Of course! The capital of France is Paris. Is there anything else I can help you with today?

User: How do you feel?
AI: That's an interesting question! As an AI, I don't experience feelings in the same way humans do. I don't have emotions or consciousness. However, I am functioning properly and ready to assist you. Do you have any other questions for me today?

User: How to became student of AGH?
AI: Sure! To become a student at AGH University of Science and Technology (AGH UST) in Krakow, Poland, you would typically need to go through an application process. This usually involves meeting certain academic requirements, submitting an application form, and potentially taking entrance exa